In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
import os
import logging
import numpy as np
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import yaml
from src.utils.constants import DATA_PATH, RAW_DATA_PATH, CENSUS_DATA_PATH, OUTPUT_DIR
from src.utils.data_utils import load_data_dic, outcome_mappings, get_weights_at_loc

logging.getLogger().setLevel(logging.INFO)

In [ ]:
STUDY = os.environ.get("LONGCOVID_STUDY", "test_based_control")
config = yaml.safe_load((REPO_ROOT / "configs" / f"{STUDY}.yaml").read_text())
data_path = str(Path(DATA_PATH) / config["data_path"])
raw_data_path = RAW_DATA_PATH
weights_path = str(Path(OUTPUT_DIR) / STUDY / "weights.pkl")
experiment_name = STUDY.replace("_", " ")
TABLE_OUTPUT = Path(OUTPUT_DIR) / STUDY / "tables"
TABLE_OUTPUT.mkdir(parents=True, exist_ok=True)

# Generate Summary Table

In [ ]:
###### Helper functions


def calculate_gender_proportion(matrix, column_index, total, weights=None):
    if weights is not None:
        value = np.sum(matrix[:, column_index].toarray().flatten() * weights)
        percentage = value / np.sum(weights)
        value *= total
    else:
        value = matrix[:, column_index].sum()
        percentage = value / total

    return value, percentage


def get_value_string_gender(matrix, column_index, total, weights=None):
    value, percentage = calculate_gender_proportion(
        matrix, column_index, total, weights=weights
    )
    return "{:,} ({}%)".format(round(value), round(100 * percentage))


def get_std_proportion(p):
    return np.sqrt(p * (1 - p))


def get_weighted_std(column, weights):
    column_mean = np.average(column, weights=weights)
    variance = np.average((column - column_mean) ** 2, weights=weights)
    column_std = np.sqrt(variance)
    return column_std


def calculate_smd(mean1, mean2, std1, std2, n1, n2):
    # Calculate the pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1) * std1**2 + (n2 - 1) * std2**2) / (n1 + n2 - 2))

    # Calculate the standardized mean difference (SMD)
    smd = (mean1 - mean2) / pooled_std
    return f"{smd:.2f}"


def calculate_values_age(matrix, column_index, weights=None):
    column = matrix[:, column_index].toarray().flatten()

    if np.isnan(column).any():
        mask = ~np.isnan(column)
        column = column[mask]

        if weights is not None:
            weights = weights[mask]

    column_mean = np.average(column, weights=weights)
    variance = np.average((column - column_mean) ** 2, weights=weights)
    column_std = np.sqrt(variance)
    return column_mean, column_std


def get_value_string_age(matrix, column_index, weights=None):
    column_mean, column_std = calculate_values_age(
        matrix, column_index, weights=weights
    )
    return f"{round(column_mean)} ({round(column_std, 1)})"


def calculate_age_bucket_proportion(
    matrix, column_index, bucket_min, bucket_max, total, weights=None
):
    column = matrix[:, column_index].toarray().flatten()

    weights_filtered = None
    if weights is not None:
        weights_filtered = weights.copy()

    if bucket_min is not None:
        if weights_filtered is not None:
            weights_filtered = weights_filtered[(column >= bucket_min)]
        column = column[column >= bucket_min]

    if bucket_max is not None:
        if weights_filtered is not None:
            weights_filtered = weights_filtered[(bucket_max >= column)]
        column = column[bucket_max >= column]

    if weights_filtered is not None:
        assert weights_filtered.shape[0] == column.shape[0]

    if weights_filtered is not None:
        value = np.sum(weights_filtered)
        percentage = value / np.sum(weights)
        value *= total
    else:
        value = column.shape[0]
        percentage = value / total

    return value, percentage


def get_age_bucket(matrix, column_index, bucket_min, bucket_max, total, weights=None):
    value, percentage = calculate_age_bucket_proportion(
        matrix, column_index, bucket_min, bucket_max, total, weights=weights
    )
    return "{:,} ({}%)".format(round(value), round(100 * percentage))


def read_raw_tsv(filepath, usecols=None, dtype=None):
    df = pd.read_csv(
        filepath,
        delimiter="\t",
        usecols=usecols,
        dtype=dtype,
    )
    return df


def get_cohort_patient_zips(filepaths, patient_ids_cohort, usecols=None, dtype=None):
    def read_one(filepath):
        return read_raw_tsv(filepath, usecols=usecols, dtype=dtype)

    with ThreadPoolExecutor(max_workers=min(32, len(filepaths))) as executor:
        df_all = pd.concat(list(executor.map(read_one, filepaths)), ignore_index=True)
    return df_all[df_all["PATIENT_ID"].isin(patient_ids_cohort)]


def map_census_data(zipcode, census_dict):
    return census_dict.get(zipcode, np.nan)


def get_static_covariates_df():
    static_filenames = [
        filename
        for filename in os.listdir(os.path.join(raw_data_path, "static_covariates/"))
        if "static_.tsv" in filename
    ]
    static_filepaths = [
        os.path.join(raw_data_path, "static_covariates/", filename)
        for filename in static_filenames
    ]

    print("STARTING get_cohort_patient_zips")
    df_static_cohort = get_cohort_patient_zips(
        static_filepaths,
        patient_ids_cohort,
        usecols=["PATIENT_ID", "PATIENT_ZIP3"],
        dtype={"PATIENT_ID": "uint32", "PATIENT_ZIP3": str},
    )
    print("FINISHED get_cohort_patient_zips")
    return df_static_cohort


def get_patient_census_df(
    df_static_cohort, patient_ids_cohort, control_patient_ids, treated_patient_ids
):
    print("FINISHED get_cohort_patient_zips")

    census_data_path = CENSUS_DATA_PATH
    census_cols = [
        "Mean Income",
        "white_percentage",
        "no bachelor degree",
    ]
    df_census = pd.read_csv(
        os.path.join(census_data_path, "processed_census_data.csv"), dtype={"ZIP3": str}
    )
    df_census["no bachelor degree"] = (
        1
        - df_census[
            "Estimate!!Total!!AGE BY EDUCATIONAL ATTAINMENT!!Population 25 years and over!!Bachelor's degree or higher Percentage"
        ]
    )
    df_census = df_census[["ZIP3"] + census_cols]
    df_census = df_census.set_index("ZIP3")

    for col in tqdm(census_cols):
        print(col)
        census_dict = df_census[col].to_dict()
        df_static_cohort[col] = df_static_cohort["PATIENT_ZIP3"].apply(
            lambda zipcode: map_census_data(zipcode, census_dict)
        )

    df_static_cohort = df_static_cohort[["PATIENT_ID"] + census_cols]

    df_patients_grouped = df_static_cohort.groupby("PATIENT_ID").mean().reset_index()

    print("PRINTING length of patient dataframe and number of nan values")
    print("Total Patients: {}".format(len(df_patients_grouped)))
    print(
        "Missing Mean Income: {}".format(
            len(df_patients_grouped[df_patients_grouped["Mean Income"].isna()])
        )
    )
    print(
        "Missing Race: {}".format(
            len(df_patients_grouped[df_patients_grouped["white_percentage"].isna()])
        )
    )
    print(
        "Missing Education: {}".format(
            len(df_patients_grouped[df_patients_grouped["no bachelor degree"].isna()])
        )
    )

    df_patients_grouped_control = df_patients_grouped[
        df_patients_grouped["PATIENT_ID"].isin(control_patient_ids)
    ]
    df_patients_grouped_treated = df_patients_grouped[
        df_patients_grouped["PATIENT_ID"].isin(treated_patient_ids)
    ]

    return df_patients_grouped, df_patients_grouped_control, df_patients_grouped_treated


def data_to_latex_table(data, index):
    df = pd.DataFrame(data, index=index)

    latex_str = df.to_latex(
        header=True, index=True, column_format="l|ccccc", escape=False
    )
    latex_str = latex_str.replace("%", "\\%")

    subheader = "& & & Unweighted & Weighted & \\\\"
    subheader += (
        "& (n={:,}) & (n={:,}) & (n={:,}) & (n={:,}) & & (n=326,569,308)\\\\".format(
            total_cohort, total_treated, total_control, total_control
        )
    )
    latex_str = (
        latex_str[: latex_str.index("\\midrule")]
        + subheader
        + latex_str[latex_str.index("\\midrule") :]
    )
    latex_str = latex_str.replace("Control Unweighted", "Control")
    latex_str = latex_str.replace("Control Weighted", "Control")
    latex_str = latex_str.replace(
        "Standardized Mean Difference (SMD)", "SMD\\tnote{\\S}"
    )

    # \tnote{\S}

    index_headers = ["Demographics", "Visits", "ICD Codes"]
    index_subheaders = [
        "Male",
        "Non-White or Hispanic",
        "Age, years",
        "Mean Income",
        "Less than Bachelor Degree",
        "Ambulance",
        "Emergency Room",
        "Home",
        "Inpatient",
        "Laboratory",
        "Non-hospital_institution",
        "Other",
        "Outpatient",
        "Pharmacy",
        "Total",
    ]
    index_subsubheaders = ["Under 18", "18-64", "65+"]

    for index_header in index_headers:
        latex_str = latex_str.replace(index_header, "\\textbf{" + index_header + "}")

    for index_subheader in index_subheaders:
        latex_str = latex_str.replace(
            index_subheader, "\\hspace{0.5em}" + index_subheader
        )

    for index_subsubheader in index_subsubheaders:
        latex_str = latex_str.replace(
            index_subsubheader, "\\hspace{1.0em}" + index_subsubheader
        )

    latex_str = latex_str.replace("Mean Income", "Mean Income\\tnote{*}")

    #                                  "centering\n" + caption + "\n\\begin{threeparttable}")

    latex_str = latex_str.replace(
        "end{tabular}",
        """end{tabular}\n\\begin{tablenotes}\n\\item[*] Mean Income values are adjusted to 2020 inflation dollars.\n\\end{tablenotes}\n\\end{threeparttable}""",
    )

    if "l|cccc" in latex_str:
        latex_str = latex_str.replace("l|ccccc", "l|ccccc|c")
        latex_str = latex_str.replace("Reference", "Reference\\tnote{$\\dagger$}")
        latex_str = latex_str.replace(
            "begin{tablenotes}",
            "begin{tablenotes}\n\\item[\\S] Standardized mean difference computed between exposed and control weighted groups.",
        )

    latex_str = latex_str.replace(
        "begin{tablenotes}",
        "begin{tablenotes}\n\\item[$\\dagger$] Calculated on 2020 census US population.",
    )

    caption = (
        "\\caption{Summary statistics of patient demographics and healthcare system interactions, stratified by total, exposed, and control groups for the "
        + experiment_name
        + ".}"
    )
    label = experiment_name.replace(" ", "_")[:10]

    begin_string = (
        "\\begin{table}[ht]\n\\scriptsize\n\\centering"
        + caption
        + "\n\\begin{threeparttable}"
    )
    end_string = "\\label{tab:" + label + "}\n\\end{table}"

    latex_str = begin_string + latex_str + end_string

    return latex_str

### Data elements from data dic

In [ ]:
data_dic = load_data_dic(data_path)

In [ ]:
treated_x = data_dic["X"][data_dic["TREATED"] == 1]
control_x = data_dic["X"][data_dic["TREATED"] == 0]

treated_y = data_dic["Y"][data_dic["TREATED"] == 1]
control_y = data_dic["Y"][data_dic["TREATED"] == 0]

treated_patient_ids = set(data_dic["Patient_id"][data_dic["TREATED"] == 1])
control_patient_ids = set(data_dic["Patient_id"][data_dic["TREATED"] == 0])

total_cohort = data_dic["X"].shape[0]
total_treated = treated_x.shape[0]
total_control = control_x.shape[0]

print(total_cohort)
print(total_treated)
print(total_control)

In [ ]:
# Get weights
weights, weight_tokens = get_weights_at_loc(weights_path)
patient_tokens_control = data_dic["Patient_token"][data_dic["TREATED"] == 0]

weights_dict = {weight_tokens[i]: weights[i] for i in range(len(weight_tokens))}
df = pd.DataFrame({"patient_token": patient_tokens_control})
df["weight"] = df["patient_token"].map(weights_dict).fillna(0.0)
weights_control = df["weight"].values

patient_token_to_patient_id = {
    data_dic["Patient_token"][i]: data_dic["Patient_id"][i]
    for i in range(len(data_dic["Patient_token"]))
}
df["patient_id"] = df["patient_token"].map(patient_token_to_patient_id)

In [ ]:
data = {
    "Total": [],
    "Exposed": [],
    "Control Unweighted": [],
    "Control Weighted": [],
    "Standardized Mean Difference (SMD)": [],
}

index = []

index.append("Demographics")
data["Total"].append("")
data["Exposed"].append("")
data["Control Unweighted"].append("")
data["Control Weighted"].append("")
data["Standardized Mean Difference (SMD)"].append("")


# Gender
male_index = data_dic["X_labels"].index("GENDER_M")
index.append("Male")
data["Total"].append(get_value_string_gender(data_dic["X"], male_index, total_cohort))
data["Exposed"].append(get_value_string_gender(treated_x, male_index, total_treated))
data["Control Unweighted"].append(
    get_value_string_gender(control_x, male_index, total_control)
)
data["Control Weighted"].append(
    get_value_string_gender(
        control_x, male_index, total_control, weights=weights_control
    )
)

_, proportion_gender_treated = calculate_gender_proportion(
    treated_x, male_index, total_treated, weights=None
)
_, proportion_gender_weighted_control = calculate_gender_proportion(
    control_x, male_index, total_control, weights=weights_control
)
std_gender_treated = get_std_proportion(proportion_gender_treated)
std_gender_control = get_std_proportion(proportion_gender_weighted_control)
data["Standardized Mean Difference (SMD)"].append(
    calculate_smd(
        proportion_gender_treated,
        proportion_gender_weighted_control,
        std_gender_treated,
        std_gender_control,
        total_treated,
        total_control,
    )
)
print("FINISHED gender")

# Age
age_index = data_dic["X_labels"].index("Age")
index.append("Age, years (mean [SD])")
data["Total"].append(get_value_string_age(data_dic["X"], age_index))
data["Exposed"].append(get_value_string_age(treated_x, age_index))
data["Control Unweighted"].append(get_value_string_age(control_x, age_index))
data["Control Weighted"].append(
    get_value_string_age(control_x, age_index, weights=weights_control)
)

mean_age_treated, std_age_treated = calculate_values_age(
    treated_x, age_index, weights=None
)
mean_age_control_weighted, std_age_control_weighted = calculate_values_age(
    control_x, age_index, weights=weights_control
)
data["Standardized Mean Difference (SMD)"].append(
    calculate_smd(
        mean_age_treated,
        mean_age_control_weighted,
        std_age_treated,
        std_age_control_weighted,
        total_treated,
        total_control,
    )
)
print("FINISHED age")

# Age Bucketed
index.append("Age, years")
data["Total"].append("")
data["Exposed"].append("")
data["Control Unweighted"].append("")
data["Control Weighted"].append("")
data["Standardized Mean Difference (SMD)"].append("")

age_index = data_dic["X_labels"].index("Age")
age_buckets = [("Under 18", None, 17), ("18-64", 18, 64), ("65+", 65, None)]
for name, lower, upper in age_buckets:
    index.append(name)
    data["Total"].append(
        get_age_bucket(data_dic["X"], age_index, lower, upper, total_cohort)
    )
    data["Exposed"].append(
        get_age_bucket(treated_x, age_index, lower, upper, total_treated)
    )
    data["Control Unweighted"].append(
        get_age_bucket(control_x, age_index, lower, upper, total_control)
    )
    data["Control Weighted"].append(
        get_age_bucket(
            control_x, age_index, lower, upper, total_control, weights=weights_control
        )
    )

    _, proportion_age_bucket_treated = calculate_age_bucket_proportion(
        treated_x, age_index, lower, upper, total_treated
    )
    _, proportion_age_bucket_control_weighted = calculate_age_bucket_proportion(
        control_x, age_index, lower, upper, total_control, weights=weights_control
    )

    std_age_bucket_treated = get_std_proportion(proportion_age_bucket_treated)
    std_age_bucket_control = get_std_proportion(proportion_age_bucket_control_weighted)

    data["Standardized Mean Difference (SMD)"].append(
        calculate_smd(
            proportion_age_bucket_treated,
            proportion_age_bucket_control_weighted,
            std_age_bucket_treated,
            std_age_bucket_control,
            total_treated,
            total_control,
        )
    )

    print(f"Finished {name}")

In [ ]:
table_latex = data_to_latex_table(data, index)
print(table_latex)
(TABLE_OUTPUT / "summary_table.tex").write_text(table_latex)
pd.DataFrame(data, index=index).to_csv(TABLE_OUTPUT / "summary_table.csv")

In [ ]:
patient_ids_cohort = set(data_dic["Patient_id"])

### Elements involving census data

In [ ]:
df_static_cohort = get_static_covariates_df()

(df_patients_grouped, df_patients_grouped_control, df_patients_grouped_treated) = (
    get_patient_census_df(
        df_static_cohort, patient_ids_cohort, control_patient_ids, treated_patient_ids
    )
)

In [ ]:
patient_id_to_weight = df.set_index("patient_id")["weight"].to_dict()
df_patients_grouped_control["weight"] = df_patients_grouped_control["PATIENT_ID"].map(
    patient_id_to_weight
)

In [ ]:
white_percentage_total = df_patients_grouped["white_percentage"].mean() * 100
white_percentage_control = df_patients_grouped_control["white_percentage"].mean() * 100
white_percentage_treated = df_patients_grouped_treated["white_percentage"].mean() * 100

index.append("Non-White or Hispanic")
data["Total"].append(f"{round(100 - white_percentage_total)}%")
data["Exposed"].append(f"{round(100 - white_percentage_treated)}%")
data["Control Unweighted"].append(f"{round(100 - white_percentage_control)}%")

weighted_white_percentage = (
    100
    * (
        df_patients_grouped_control["white_percentage"]
        * df_patients_grouped_control["weight"]
    ).sum()
    / df_patients_grouped_control["weight"].sum()
)
data["Control Weighted"].append(f"{round(100 - weighted_white_percentage)}%")

# SMD
df_patients_grouped_treated_no_nan = df_patients_grouped_treated[
    ~df_patients_grouped_treated["white_percentage"].isna()
]
std_white_treated = get_weighted_std(
    df_patients_grouped_treated_no_nan["white_percentage"], None
)
n_treated = len(df_patients_grouped_treated_no_nan)

df_patients_grouped_control_no_nan = df_patients_grouped_control[
    ~df_patients_grouped_control["white_percentage"].isna()
]
std_white_weighted_control = get_weighted_std(
    df_patients_grouped_control_no_nan["white_percentage"],
    df_patients_grouped_control_no_nan["weight"],
)
n_weighted_control = df_patients_grouped_control_no_nan["weight"].sum() * len(
    df_patients_grouped_control_no_nan
)

white_smd = calculate_smd(
    white_percentage_treated / 100,
    weighted_white_percentage / 100,
    std_white_treated,
    std_white_weighted_control,
    n_treated,
    n_weighted_control,
)
data["Standardized Mean Difference (SMD)"].append(white_smd)

In [ ]:
no_degree_total = df_patients_grouped["no bachelor degree"].mean() * 100
no_degree_control = df_patients_grouped_control["no bachelor degree"].mean() * 100
no_degree_treated = df_patients_grouped_treated["no bachelor degree"].mean() * 100

index.append("Less than Bachelor Degree")
data["Total"].append(f"{round(no_degree_total)}%")
data["Exposed"].append(f"{round(no_degree_treated)}%")
data["Control Unweighted"].append(f"{round(no_degree_control)}%")

weighted_no_degree_percentage = (
    100
    * (
        df_patients_grouped_control["no bachelor degree"]
        * df_patients_grouped_control["weight"]
    ).sum()
    / df_patients_grouped_control["weight"].sum()
)
data["Control Weighted"].append(f"{round(weighted_no_degree_percentage)}%")

# SMD
df_patients_grouped_treated_no_nan = df_patients_grouped_treated[
    ~df_patients_grouped_treated["no bachelor degree"].isna()
]
std_degree_treated = get_weighted_std(
    df_patients_grouped_treated_no_nan["no bachelor degree"], None
)
n_treated = len(df_patients_grouped_treated_no_nan)

df_patients_grouped_control_no_nan = df_patients_grouped_control[
    ~df_patients_grouped_control["no bachelor degree"].isna()
]
std_degree_weighted_control = get_weighted_std(
    df_patients_grouped_control_no_nan["no bachelor degree"],
    df_patients_grouped_control_no_nan["weight"],
)
n_weighted_control = df_patients_grouped_control_no_nan["weight"].sum() * len(
    df_patients_grouped_control_no_nan
)

degree_smd = calculate_smd(
    no_degree_treated / 100,
    weighted_no_degree_percentage / 100,
    std_degree_treated,
    std_degree_weighted_control,
    n_treated,
    n_weighted_control,
)
data["Standardized Mean Difference (SMD)"].append(degree_smd)

In [ ]:
mean_income_total = df_patients_grouped["Mean Income"].mean()
mean_income_control = df_patients_grouped_control["Mean Income"].mean()
mean_income_treated = df_patients_grouped_treated["Mean Income"].mean()

index.append("Mean Income")
data["Total"].append("{:,}".format(round(mean_income_total)))
data["Exposed"].append("{:,}".format(round(mean_income_treated)))
data["Control Unweighted"].append("{:,}".format(round(mean_income_control)))

weighted_income = (
    df_patients_grouped_control["Mean Income"] * df_patients_grouped_control["weight"]
).sum() / df_patients_grouped_control["weight"].sum()

data["Control Weighted"].append("{:,}".format(round(weighted_income)))

# SMD
df_patients_grouped_treated_no_nan = df_patients_grouped_treated[
    ~df_patients_grouped_treated["Mean Income"].isna()
]
std_income_treated = get_weighted_std(
    df_patients_grouped_treated_no_nan["Mean Income"], None
)
n_treated = len(df_patients_grouped_treated_no_nan)

df_patients_grouped_control_no_nan = df_patients_grouped_control[
    ~df_patients_grouped_control["Mean Income"].isna()
]
std_income_weighted_control = get_weighted_std(
    df_patients_grouped_control_no_nan["Mean Income"],
    df_patients_grouped_control_no_nan["weight"],
)
n_weighted_control = df_patients_grouped_control_no_nan["weight"].sum() * len(
    df_patients_grouped_control_no_nan
)

income_smd = calculate_smd(
    mean_income_treated,
    weighted_income,
    std_income_treated,
    std_income_weighted_control,
    n_treated,
    n_weighted_control,
)
data["Standardized Mean Difference (SMD)"].append(income_smd)

In [ ]:
table_latex = data_to_latex_table(data, index)
print(table_latex)
(TABLE_OUTPUT / "summary_table.tex").write_text(table_latex)
pd.DataFrame(data, index=index).to_csv(TABLE_OUTPUT / "summary_table.csv")

### Elements involving visits data

In [ ]:
outcome_to_indices_x, outcome_to_labels_x = outcome_mappings(data_dic["X_labels"])
outcome_to_indices_y, outcome_to_labels_y = outcome_mappings(data_dic["Y_labels"])

In [ ]:
def get_total_visits(X_matrix, Y_matrix, indices_x, indices_y, weights=None):

    if weights is None:
        sum_x = X_matrix[:, indices_x].sum()
        sum_y = Y_matrix[:, indices_y].sum()
        return str(round((sum_x + sum_y) / X_matrix.shape[0], 2))

    else:
        column_sum = (
            np.asarray(X_matrix[:, indices_x].sum(axis=1)).squeeze()
            + np.asarray(Y_matrix[:, indices_y].sum(axis=1)).squeeze()
        )
        weighted_sum = column_sum * weights

        return str(round(weighted_sum.sum() / weights.sum(), 2))

In [ ]:
all_visit_types = {
    "Ambulance_Visit",
    "Emergency_Room_Visit",
    "Home_Visit",
    "Inpatient_Visit",
    "Laboratory_Visit",
    "Non-hospital_institution_Visit",
    "Other_Visit",
    "Outpatient_Visit",
    "Pharmacy Visit",
}
individual_visit_types = {
    "Inpatient_Visit",
    "Outpatient_Visit",
    "Emergency_Room_Visit",
}

index.append("Visits (mean / patient)")
data["Total"].append("")
data["Exposed"].append("")
data["Control Unweighted"].append("")
data["Control Weighted"].append("")
data["Standardized Mean Difference (SMD)"].append("")

all_visit_indices_x = []
all_visit_indices_y = []

for visit_type in all_visit_types:
    print(visit_type)
    visit_col_indices_x = outcome_to_indices_x[visit_type]
    visit_col_indices_y = outcome_to_indices_y[visit_type]

    all_visit_indices_x.append(visit_col_indices_x)
    all_visit_indices_y.append(visit_col_indices_y)

    if visit_type not in individual_visit_types:
        continue

    index.append(visit_type.replace("_", " ")[: visit_type.index("Visit") - 1])
    data["Total"].append(
        get_total_visits(
            data_dic["X"], data_dic["Y"], visit_col_indices_x, visit_col_indices_y
        )
    )
    data["Exposed"].append(
        get_total_visits(treated_x, treated_y, visit_col_indices_x, visit_col_indices_y)
    )
    data["Control Unweighted"].append(
        get_total_visits(control_x, control_y, visit_col_indices_x, visit_col_indices_y)
    )

    data["Control Weighted"].append(
        get_total_visits(
            control_x,
            control_y,
            visit_col_indices_x,
            visit_col_indices_y,
            weights=weights_control,
        )
    )
    data["Standardized Mean Difference (SMD)"].append("")


all_visit_indices_x_np = np.concatenate(all_visit_indices_x)
all_visit_indices_y_np = np.concatenate(all_visit_indices_y)
index.append("Total")
data["Total"].append(
    get_total_visits(
        data_dic["X"], data_dic["Y"], all_visit_indices_x_np, all_visit_indices_y_np
    )
)
data["Exposed"].append(
    get_total_visits(
        treated_x, treated_y, all_visit_indices_x_np, all_visit_indices_y_np
    )
)
data["Control Unweighted"].append(
    get_total_visits(
        control_x, control_y, all_visit_indices_x_np, all_visit_indices_y_np
    )
)
data["Control Weighted"].append(
    get_total_visits(
        control_x,
        control_y,
        all_visit_indices_x_np,
        all_visit_indices_y_np,
        weights=weights_control,
    )
)
data["Standardized Mean Difference (SMD)"].append("")

print("Total")

In [ ]:
table_latex = data_to_latex_table(data, index)
print(table_latex)
(TABLE_OUTPUT / "summary_table.tex").write_text(table_latex)
pd.DataFrame(data, index=index).to_csv(TABLE_OUTPUT / "summary_table.csv")

### Reference, US Census

In [ ]:
data["Reference"] = [""] * len(index)

data["Reference"][index.index("Non-White or Hispanic")] = "43%"
data["Reference"][index.index("Less than Bachelor Degree")] = "67%"
data["Reference"][index.index("Mean Income")] = "90,981"
data["Reference"][index.index("Male")] = "160,818,530 (49%)"
data["Reference"][index.index("Under 18")] = "73,296,738 (22%)"
data["Reference"][index.index("65+")] = "52,362,817 (16%)"
data["Reference"][index.index("18-64")] = "200,909,753 (62%)"

In [ ]:
for i in range(len(data["Standardized Mean Difference (SMD)"])):
    if data["Standardized Mean Difference (SMD)"][i] == "nan":
        data["Standardized Mean Difference (SMD)"][i] = "0.00"

In [ ]:
table_latex = data_to_latex_table(data, index)
print(table_latex)
(TABLE_OUTPUT / "summary_table.tex").write_text(table_latex)
pd.DataFrame(data, index=index).to_csv(TABLE_OUTPUT / "summary_table.csv")